# Syteline (Infor CSI) -> Eventhouse Ingestion

Metadata-driven ingestion from Infor CloudSuite Industrial (Syteline) into Microsoft
Fabric: pulls entity data via the **ION REST IDO API** and writes it straight into an
**Eventhouse** table with **Kusto streaming ingestion**. Job definitions and per-entity
incremental watermarks live in a Warehouse control table, so adding an entity is a
control-row registration - not a notebook edit.

This is a **pure Python notebook** (no Spark), and it lands the data itself: a single
`POST` per batch, answered only once the rows are committed and queryable. There is no
Eventstream, no CloudEvents envelope, no schema registry and no SDK on the path - see
*Transport* below, which also records why the CloudEvents producer this notebook used to
carry was retired rather than kept as an option.

The pattern generalizes to any REST source: swap the *IDO load & transform* cell for your
source's paging API and keep the control-plane and transport cells unchanged.

**Usage:**
1. Create the control-plane objects in a Fabric Warehouse (see *Control-plane contract*)
   and register at least one entity.
2. Store the four ION credentials in Azure Key Vault (see *Configuration*).
3. Create the landing table in the KQL database, then clear its streaming-ingestion schema
   cache - `.create-merge` alone leaves the cache stale and new columns land empty:
   ```kusto
   .create-merge table ['<Entity>_<SchemaVersion>'] (<Column>: string, ...)
   .clear table ['<Entity>_<SchemaVersion>'] cache streamingingestion schema
   ```
4. Fill the parameters cell and the `<Placeholders>` in Configuration - or leave the
   parameters blank and define matching variables in a workspace **Variable Library**;
   blanks resolve from its active value set at run time.
5. Run with `DRY_RUN = True` first: it loads, transforms, and prints one sample row and the
   table it would land in, without sending anything, advancing watermarks, or logging.

**Auth:** ION uses an OAuth2 resource-owner (password) grant - SAAK/SASK service-account
keys as username/password plus a backing-service client id/secret, all four values from
the tenant's `.ionapi` credential file, all four in Key Vault. Everything on the Fabric
side - the Warehouse over TDS and the Eventhouse ingest - uses the notebook identity's own
Entra token, so those four are the only secrets on the path. The identity needs
`Table ingestor` on the target KQL database.

**Scheduling:** invoke from a Data Pipeline Notebook activity that passes every parameter
explicitly (e.g. from a Variable Library `libraryVariables` block), including
`DRY_RUN = false` and `ENTITY_FILTER`, so scheduled runs go live without editing this
notebook. Parameter-complete runs never call `notebookutils.variableLibrary` - which
matters because that API has no service-principal support.

## Control-plane contract

One Warehouse schema (`ingest`) carries the control table, a run log, and their procs.
The notebook reads `ingest.Control`, inserts `ingest.RunLog` telemetry via
`ingest.usp_LogRun`, and advances `LastWatermark` directly (sequential per-entity
updates - safe because this notebook is not parallel).

```sql
CREATE TABLE ingest.Control (
    SourceSystemName varchar(20)  NOT NULL, -- e.g. 'Syteline-ION'
    SourceObjectName varchar(50)  NOT NULL, -- IDO name; also the landing table's base name
    SchemaVersion    varchar(10)  NOT NULL, -- landing table name suffix, e.g. 'v1'
    SourceWatermark  varchar(50)  NOT NULL, -- IDO property filtered for the incremental window
    SourceOrderBy    varchar(200) NULL,
    SourceFilter     varchar(max) NULL,     -- extra IDO filter ANDed onto the watermark window
    FieldMap         varchar(max) NOT NULL, -- JSON: [{"OutputFieldName": "...", "Sources": ["P(Prop)", "literal"]}]
    LastWatermark    datetime2(3) NULL,     -- run state: advanced on success only
    IsActive         bit          NOT NULL
);
ALTER TABLE ingest.Control ADD CONSTRAINT PK_IngestControl
    PRIMARY KEY NONCLUSTERED (SourceSystemName, SourceObjectName);
```

Rows land in `<SourceObjectName>_<SchemaVersion>` in the KQL database - composed by the
notebook, from these two columns. Every landing column is `string`; type downstream.

`ingest.usp_LogRun` inserts one row per entity per run (`@PipelineRunId`,
`@SourceSystemName`, `@SourceObjectName`, `@Status`, `@RecordsFetched`, `@RecordsSent`,
`@DurationSeconds`, `@Watermark`, `@FailureMessage`). `@RecordsSent` is the ingest
receipt's `ConsumedRecordsCount` - records *committed*, not records handed to a sink.
Register control rows through an idempotent stored procedure rather than raw INSERTs, and
never overwrite `LastWatermark` on re-registration - it is run state, not config; rewinding
it re-copies the whole window.

`FieldMap` sources: `P(PropertyName)` resolves to the IDO property; any other string is a
literal. Sources concatenate in order into the output field - all values emit as strings.
**The request is derived from the sources, not from the field list**: the properties asked
of the IDO are the sorted, deduplicated set of names across every `P(Name)`, so a field
built only from literals adds a column and asks the source for nothing.

A malformed source is silently a literal. `P(Item` or a bare `Item` is an error nowhere -
the column then carries that same text on every row while the property is never requested,
which fills the table successfully with the wrong data. `local-cli/ido.sh` reads this same
control row to reconstruct the request, and its `-r` flag shows what actually came back.

## Parameters (pipeline-overridable)

Mark this as the notebook's **parameter cell** after importing into Fabric (cell toolbar
-> *Toggle parameter cell*; the tag does not survive metadata scrubbing). A pipeline
Notebook activity overrides these at submission. Blank values resolve from the workspace
Variable Library named in Configuration (active value set), so an interactive run in any
workspace picks up that workspace's config with no edits here. `DRY_RUN` ships closed
for interactive safety; the pipeline passes `false` plus the run-scoping knobs for live
scheduled runs.

In [ ]:
# DRY_RUN  True (default) - load + transform + report per-entity counts and one sample
#          row; send nothing, advance no watermarks, log nothing.
#          The pipeline passes false for live scheduled runs.
DRY_RUN = True
# Comma-separated SourceObjectName list restricting the run (e.g. "<Entity>,<Entity>");
# blank = every active control row. A string, not a list - notebook parameters are
# scalars only - parsed to the list the run loop consumes in Configuration.
ENTITY_FILTER = ""
# Everything below: blank = resolved from the Variable Library in Configuration (variable
# named in the comment). Pipelines should pass all of these explicitly - parameter-complete
# runs never hit the variableLibrary API (same-workspace only, no SP support).
WAREHOUSE_SQL_SERVER = ""  # WarehouseSqlEndpoint - <guid>.datawarehouse.fabric.microsoft.com
WAREHOUSE_DB = ""          # WarehouseName - item display name (TDS Initial Catalog rule)
PIPELINE_RUN_ID = ""       # @pipeline().RunId when pipeline-invoked; blank mints a GUID
KEY_VAULT_URI = ""         # AzureKeyVaultUri
KUSTO_URI = ""             # KustoUri - the Eventhouse QUERY URI (https://<guid>.kusto.
                           # fabric.microsoft.com), NOT an ingest- host: streaming
                           # ingestion posts to the engine, queued ingestion to ingest-.
KUSTO_DATABASE = ""        # KustoDatabaseName - KQL database holding the landing tables

## Configuration

Blank parameters resolve here from the workspace Variable Library's active value set.
Secrets live in Key Vault and are fetched with `notebookutils.credentials.getSecret` -
nothing secret is stored in the notebook or the control table.

All four Key Vault secrets are **unsuffixed**: one ION tenant serves every Fabric stage, so
the source credentials are shared rather than environment-specific. Retiring the Eventstream
SAS key removed the last per-environment secret this notebook needed, and the
`ENVIRONMENT_SUFFIX` parameter with it. What still moves per stage - the Warehouse endpoint,
the Eventhouse URI and database - moves through the Variable Library's value sets, which is
where stage-specific *configuration* belongs.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────
# --- Environment resolution (hard fail) ---
# Any parameter left blank resolves from this workspace's Variable Library (active value
# set). Lazy: a fully parameterized (pipeline) run makes no variableLibrary calls at all.
import notebookutils  # imported early (before the Imports cell) for resolution

VARIABLE_LIBRARY_NAME = "<VariableLibraryName>"


def vl_lookup(variable_name: str):
    """Resolve one Variable Library variable from the workspace's active value set.

    Args:
        variable_name: Variable name in the library (case-sensitive).

    Returns:
        The typed variable value.

    Raises:
        RuntimeError: If the library or variable can't be resolved here.
    """
    try:
        return notebookutils.variableLibrary.get(
            f"$(/**/{VARIABLE_LIBRARY_NAME}/{variable_name})"
        )
    except Exception as exc:
        raise RuntimeError(
            f"Variable Library lookup failed for '{variable_name}'. Check that "
            f"'{VARIABLE_LIBRARY_NAME}' exists in this workspace and this runtime supports "
            "notebookutils.variableLibrary - or fill the value in the parameters cell."
        ) from exc


WAREHOUSE_SQL_SERVER = WAREHOUSE_SQL_SERVER or vl_lookup("WarehouseSqlEndpoint")
WAREHOUSE_DB = WAREHOUSE_DB or vl_lookup("WarehouseName")
KEY_VAULT_URI = KEY_VAULT_URI or vl_lookup("AzureKeyVaultUri")
KUSTO_URI = KUSTO_URI or vl_lookup("KustoUri")
KUSTO_DATABASE = KUSTO_DATABASE or vl_lookup("KustoDatabaseName")

# --- Run scoping ---
# STRICT: True raises after the summary if any entity failed (CI / orchestration).
STRICT = False
# ENTITY_FILTER arrives as a comma-separated string from the parameters cell; parse to
# the list the run loop consumes. Empty = every active control row. A blank pipeline
# parameter is injected as None (Fabric nulls empty string parameters), so guard it.
ENTITY_FILTER: list = [e.strip() for e in (ENTITY_FILTER or "").split(",") if e.strip()]

# --- Control plane ---
SOURCE_SYSTEM = "Syteline-ION"
INGEST_TABLE = "ingest.Control"
DEFAULT_WATERMARK = "1900-01-01T00:00:00"  # backfill start when LastWatermark is NULL

# --- Key Vault secret names (names only - values stay in the vault) ---
# All four are unsuffixed: one ION tenant serves every Fabric stage, so these are shared
# rather than environment-specific. Retiring the Eventstream SAS key removed the last
# environment-suffixed secret this notebook needed - and with it the ENVIRONMENT_SUFFIX
# parameter. A source whose credentials really do differ per stage should reintroduce a
# suffix here rather than splitting the vault.
SECRET_ION_CLIENT_ID = "ion-client-id"          # .ionapi "ci"
SECRET_ION_CLIENT_SECRET = "ion-client-secret"  # .ionapi "cs"
SECRET_ION_SAAK = "ion-saak"                    # .ionapi "saak" (service account access key)
SECRET_ION_SASK = "ion-sask"                    # .ionapi "sask" (service account secret key)

# --- ION API (non-secret; values come from the tenant's .ionapi credential file) ---
ION_TENANT = "<IonTenantId>"                                # .ionapi "ti"
ION_API_BASE = "https://mingle-ionapi.inforcloudsuite.com"  # .ionapi "iu"
ION_TOKEN_URL = f"https://mingle-sso.inforcloudsuite.com/{ION_TENANT}/as/token.oauth2"
ION_IDO_SUITE = "CSI"  # IDO endpoints live under {iu}/{ti}/CSI
# Syteline (Mongoose) configuration name - sent as the X-Infor-MongooseConfig header on
# every IDO call.
ION_MONGOOSE_CONFIG = "<MongooseConfigName>"
# Syteline stores RecordDate in server-local time; shift the incremental window to match.
SYTELINE_UTC_OFFSET_HOURS = 0

# --- Kusto streaming ingestion (see the Transport cell for the protocol) ---
# Audience for the Entra token. The same one the Eventhouse query endpoint takes: the
# ingest POST and the .clear management command both go to KUSTO_URI, so there is exactly
# one auth surface on this path and no SAS to rotate.
KUSTO_TOKEN_AUDIENCE = "https://kusto.kusto.windows.net"
# Documented per-request cap for streaming ingestion. A single record larger than this
# raises rather than being silently split.
KUSTO_STREAM_LIMIT_BYTES = 4 * 1024 * 1024
# 429 rate-limited, 503/504 transient, 520 the cold-table window in which a freshly
# created table refuses every ingest for ~80 s. EntityNotFound is a 400 and deliberately
# absent: a genuinely missing table returns the same status, and retrying would stall a
# real misconfiguration for minutes per entity. It gets one schema-cache clear instead.
KUSTO_RETRY_STATUS = (429, 503, 504, 520)
KUSTO_INGEST_ATTEMPTS = 7
KUSTO_RETRY_BASE_DELAY = 4.0   # seconds; doubles per attempt, plus jitter
KUSTO_RETRY_MAX_DELAY = 60.0
# Entra tokens last ~60 min; re-mint at 45 to leave headroom on a long window.
KUSTO_TOKEN_REFRESH_AFTER_SECONDS = 2700
# Appended to an EntityNotFound that survived the cache clear - both commands, because
# .create-merge alone leaves the streaming-ingestion schema cache stale.
KUSTO_TABLE_MISSING_HINT = (
    "\n    Create it, then clear the streaming-ingestion schema cache:"
    "\n      .create-merge table ['{table}'] (<Column>: string, ...)"
    "\n      .clear table ['{table}'] cache streamingingestion schema"
)
# Cap on the printed sample body. Wide entities serialize to several KB per record; the
# shape is legible long before that and the run log stays readable.
SAMPLE_ROW_MAX_CHARS = 4000

# Records per IDO page; the load loop follows MoreRowsExist/Bookmark until the window drains.
IDO_PAGE_SIZE = 5000


# Fail fast on unfilled configuration - a gap would otherwise surface as an opaque
# auth/DNS error mid-run. Blank, "FILL-ME" (Variable Library can't hold empty strings, so
# use a sentinel there), and <Placeholder> values all count as unfilled.
def _unfilled(value) -> bool:
    text = str(value)
    return not value or text == "FILL-ME" or (text.startswith("<") and text.endswith(">"))


_required = {
    "WAREHOUSE_SQL_SERVER": WAREHOUSE_SQL_SERVER,
    "WAREHOUSE_DB": WAREHOUSE_DB,
    "KEY_VAULT_URI": KEY_VAULT_URI,
    "KUSTO_URI": KUSTO_URI,
    "KUSTO_DATABASE": KUSTO_DATABASE,
    "ION_TENANT": ION_TENANT,
    "ION_MONGOOSE_CONFIG": ION_MONGOOSE_CONFIG,
}
_missing = [name for name, value in _required.items() if _unfilled(value)]
if _missing:
    raise RuntimeError(
        f"Unfilled configuration: {', '.join(_missing)}. Fill the parameters cell and the "
        "<Placeholders> above, or define the matching Variable Library variables."
    )

# Canonical result buckets - populated by the run loop below.
results = {
    "succeeded": [],  # list[str] of entity names
    "skipped":   [],  # list[dict]: {"name": str, "reason": str}
    "failed":    [],  # list[dict]: {"name": str, "error": str}
}

## Imports

Standard library plus `pandas`, `pyodbc` and `requests` - all present in the default pure
Python runtime. **Nothing self-installs.** The previous transport needed `azure-eventhub`,
which is not in the runtime and which pure Python notebooks cannot supply through an
Environment item, so the import used to `pip install` into the session on first failure
(~15-30 s on a cold session). Streaming ingestion is plain `requests` against a REST
endpoint, so that whole branch is gone along with the cold-start cost.

In [ ]:
import json
import random
import struct
import time
import urllib.parse
import uuid
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
import pyodbc
import requests

## Authentication (hard fail)

Fetches the four ION credentials from Key Vault and acquires an ION bearer token. ION uses
an OAuth2 resource-owner grant: SAAK/SASK as username/password plus the backing-service
client id/secret.

**That is the only secret material on this path.** The Warehouse connection uses the
notebook identity's Entra token over TDS, and so does the Kusto ingest - same identity,
same kind of token, no SAS key, no connection string, nothing per-environment to rotate.
The identity needs `Table ingestor` (or Database Ingestor) on the target KQL database, plus
whatever the control Warehouse grants it.

In [ ]:
def get_secret(name: str) -> str:
    """Fetch a secret from Key Vault, raising if it is missing or empty.

    Args:
        name: Secret name in the vault at KEY_VAULT_URI.

    Returns:
        The secret value.

    Raises:
        RuntimeError: If the secret cannot be fetched or is empty.
    """
    value = notebookutils.credentials.getSecret(KEY_VAULT_URI, name)
    if not value:
        raise RuntimeError(f"Key Vault secret '{name}' is empty or missing at {KEY_VAULT_URI}")
    return value


def acquire_ion_token() -> str:
    """Acquire an ION API bearer token via the OAuth2 password (SAAK/SASK) grant.

    Returns:
        The access token string.

    Raises:
        RuntimeError: On a non-200 token response, with status and body excerpt.
    """
    response = requests.post(
        ION_TOKEN_URL,
        data={
            "grant_type": "password",
            "username": get_secret(SECRET_ION_SAAK),
            "password": get_secret(SECRET_ION_SASK),
            "client_id": get_secret(SECRET_ION_CLIENT_ID),
            "client_secret": get_secret(SECRET_ION_CLIENT_SECRET),
        },
        timeout=60,
    )
    if response.status_code != 200:
        raise RuntimeError(
            f"ION token request failed: HTTP {response.status_code} - {response.text[:300]}"
        )
    token = response.json().get("access_token")
    if not token:
        raise RuntimeError("ION token response contained no access_token")
    return token


def ion_get(url: str, params: Dict[str, Any]) -> requests.Response:
    """GET against the ION API, re-acquiring the token once on a 401.

    A full backfill can outlive a single ION token (~2 h lifetime), so a mid-run 401
    triggers one transparent refresh + retry.

    Args:
        url: Full request URL.
        params: Query parameters.

    Returns:
        The (possibly retried) response; status handling stays with the caller.
    """
    response = ion_session.get(url, params=params, timeout=300)
    if response.status_code == 401:
        ion_session.headers["Authorization"] = f"Bearer {acquire_ion_token()}"
        response = ion_session.get(url, params=params, timeout=300)
    return response


ion_session = requests.Session()
ion_session.headers.update({
    "Authorization": f"Bearer {acquire_ion_token()}",
    "Accept": "application/json",
    "X-Infor-MongooseConfig": ION_MONGOOSE_CONFIG,  # required by Mongoose on every IDO call
})
# No sink credential to assemble: the Kusto ingest mints its own Entra token per send,
# from this notebook's identity, at the point of use.
print(f"ION token acquired; ingest target {KUSTO_URI} / {KUSTO_DATABASE}.")

## Control plane: job definitions + watermarks (hard fail)

`ingest.Control` is the source of truth: one row per entity carrying the IDO name (which
also names the landing table), schema version, watermark property, and the `FieldMap`
JSON. `LastWatermark` is runtime state this notebook advances on success. Warehouse
access is via **pyodbc + an Entra token** (`notebookutils.credentials.getToken` with the
SQL/TDS audience `https://database.windows.net/`, injected as `SQL_COPT_SS_ACCESS_TOKEN`).

In [ ]:
def parse_field_map(raw: str, entity: str) -> List[Dict[str, Any]]:
    """Parse and validate one entity's FieldMap JSON.

    Args:
        raw: JSON array of {OutputFieldName, Sources[]} objects.
        entity: Entity name, for error messages.

    Returns:
        The parsed field list.

    Raises:
        RuntimeError: If the JSON is invalid or the shape is wrong.
    """
    try:
        fields = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Invalid FieldMap JSON for '{entity}': {exc}") from exc
    if not isinstance(fields, list) or not fields:
        raise RuntimeError(f"FieldMap for '{entity}' must be a non-empty JSON array")
    for field in fields:
        if "OutputFieldName" not in field or "Sources" not in field:
            raise RuntimeError(
                f"FieldMap entry for '{entity}' is missing OutputFieldName/Sources: {field}"
            )
    return fields


def wh_connect() -> pyodbc.Connection:
    """Open a pyodbc connection to the control-plane Warehouse with an Entra token.

    Token audience is the SQL/TDS resource (https://database.windows.net/); the token is
    injected pre-auth via SQL_COPT_SS_ACCESS_TOKEN (attr 1256) as a UTF-16-LE
    length-prefixed struct. A fresh connection per operation keeps long backfills immune
    to token expiry.

    Returns:
        An open connection; caller commits writes.

    Raises:
        RuntimeError: If no SQL Server ODBC driver is installed in the runtime.
    """
    driver = next(
        (d for d in sorted(pyodbc.drivers(), reverse=True)
         if "ODBC Driver" in d and "SQL Server" in d),
        None,
    )
    if driver is None:
        raise RuntimeError(f"No SQL Server ODBC driver found; installed: {pyodbc.drivers()}")
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    return pyodbc.connect(
        f"Driver={{{driver}}};Server=tcp:{WAREHOUSE_SQL_SERVER},1433;"
        f"Database={WAREHOUSE_DB};Encrypt=Yes;TrustServerCertificate=No",
        attrs_before={1256: token_struct},  # SQL_COPT_SS_ACCESS_TOKEN
    )


def wh_query(sql: str) -> pd.DataFrame:
    """Run a SELECT against the Warehouse and return the result as a DataFrame.

    Args:
        sql: The T-SQL statement.

    Returns:
        DataFrame with the statement's result set (empty if no rows).
    """
    with wh_connect() as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        columns = [d[0] for d in cursor.description]
        return pd.DataFrame.from_records(
            [tuple(row) for row in cursor.fetchall()], columns=columns
        )


def wh_execute(sql: str) -> int:
    """Run a write statement against the Warehouse and commit it.

    Args:
        sql: The T-SQL statement.

    Returns:
        Number of rows affected.
    """
    with wh_connect() as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        affected = cursor.rowcount
        conn.commit()
        return affected


def load_ingest_control() -> List[Dict[str, Any]]:
    """Read active ingest jobs (definitions + watermarks) from the control table.

    Returns:
        One dict per entity: entity, schema_version, watermark_property, order_by,
        filter, fields, last_watermark (datetime or None).

    Raises:
        RuntimeError: If no active rows exist or a FieldMap fails validation.
    """
    frame = wh_query(
        f"""
        SELECT
              SourceObjectName
            , SchemaVersion
            , SourceWatermark
            , SourceOrderBy
            , SourceFilter
            , FieldMap
            , LastWatermark
        FROM {INGEST_TABLE}
        WHERE IsActive = 1
            AND SourceSystemName = '{SOURCE_SYSTEM}'
        ORDER BY SourceObjectName;
        """
    )
    if frame is None or len(frame) == 0:
        raise RuntimeError(
            f"No active rows in {INGEST_TABLE} for SourceSystemName = '{SOURCE_SYSTEM}'. "
            "Register at least one entity first (see the control-plane contract above)."
        )
    jobs: List[Dict[str, Any]] = []

    def null_safe(value: Any) -> Any:
        """Collapse pandas NULL representations (None, NaT, NaN) to None."""
        return None if pd.isna(value) else value

    for row in frame.itertuples(index=False):
        last_watermark = null_safe(row.LastWatermark)
        if last_watermark is not None and hasattr(last_watermark, "to_pydatetime"):
            last_watermark = last_watermark.to_pydatetime()
        jobs.append({
            "entity": row.SourceObjectName,
            "schema_version": row.SchemaVersion,
            "watermark_property": row.SourceWatermark,
            "order_by": null_safe(row.SourceOrderBy),
            "filter": null_safe(row.SourceFilter),
            "fields": parse_field_map(row.FieldMap, row.SourceObjectName),
            "last_watermark": last_watermark,
        })
    return jobs


def set_watermark(entity: str, watermark: datetime) -> None:
    """Advance one entity's watermark (called only on success, never in DRY_RUN).

    Sequential per-entity UPDATEs are safe here because this notebook processes entities
    one at a time; a parallel orchestrator would need a set-based advance instead to
    avoid Warehouse write-write conflicts.

    Args:
        entity: SourceObjectName to update.
        watermark: New watermark value (UTC-naive).

    Raises:
        RuntimeError: If the UPDATE did not affect exactly one row.
    """
    safe_entity = entity.replace("'", "''")
    affected = wh_execute(
        f"""
        UPDATE {INGEST_TABLE}
        SET LastWatermark = '{watermark.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]}'
        WHERE SourceSystemName = '{SOURCE_SYSTEM}'
            AND SourceObjectName = '{safe_entity}';
        """
    )
    if affected != 1:
        raise RuntimeError(f"Watermark update for '{entity}' affected {affected} rows")


RUN_ID = PIPELINE_RUN_ID or str(uuid.uuid4())


def log_run(
    entity: str,
    status: str,
    records_fetched: Optional[int] = None,
    records_sent: Optional[int] = None,
    duration_seconds: Optional[int] = None,
    watermark: Optional[datetime] = None,
    failure_message: Optional[str] = None,
) -> None:
    """Insert one ingest.RunLog row via ingest.usp_LogRun.

    Telemetry only - a logging failure prints a warning but never fails the entity
    (the run outcome is already captured in `results` and the watermark state).

    Args:
        entity: SourceObjectName the row describes.
        status: SUCCESS, NOROWS, or FAILED.
        records_fetched: Records returned by the IDO load.
        records_sent: Records committed to the landing table (ConsumedRecordsCount).
        duration_seconds: Wall-clock seconds for the entity.
        watermark: Value the entity's watermark advanced to (success only).
        failure_message: Exception text for FAILED rows.
    """
    def sql_literal(value: Any) -> str:
        if value is None:
            return "NULL"
        if isinstance(value, int):
            return str(value)
        if isinstance(value, datetime):
            return f"'{value.strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3]}'"
        return "'" + str(value).replace("'", "''") + "'"

    try:
        wh_execute(
            f"""
            EXEC ingest.usp_LogRun
                  @PipelineRunId = '{RUN_ID}'
                , @SourceSystemName = '{SOURCE_SYSTEM}'
                , @SourceObjectName = {sql_literal(entity)}
                , @Status = {sql_literal(status)}
                , @RecordsFetched = {sql_literal(records_fetched)}
                , @RecordsSent = {sql_literal(records_sent)}
                , @DurationSeconds = {sql_literal(duration_seconds)}
                , @Watermark = {sql_literal(watermark)}
                , @FailureMessage = {sql_literal(failure_message[:4000] if failure_message else None)};
            """
        )
    except Exception as exc:
        print(f"  WARNING: RunLog insert failed for {entity}: {exc}")


job_definitions = load_ingest_control()
print(f"Loaded {len(job_definitions)} active ingest jobs from {INGEST_TABLE} "
      f"(run {RUN_ID}): {', '.join(j['entity'] for j in job_definitions)}")

## IDO load & transform

GET `{iu}/{tenant}/CSI/IDORequestService/ido/load/{ido}` with `loadType=NEXT`, paging via
the response's `MoreRowsExist` / `Bookmark`, and the `X-Infor-MongooseConfig` header on
every call. This is the source-specific cell: to reuse the pattern for a different REST
source, replace it and keep everything else.

In [ ]:
def format_syteline_timestamp(value: datetime) -> str:
    """Format a watermark for an IDO filter clause.

    Syteline runs on SQL Server ``datetime`` (millisecond precision) in server-local
    time; SYTELINE_UTC_OFFSET_HOURS shifts the incremental window to compensate.

    Args:
        value: UTC-naive watermark timestamp.

    Returns:
        Timestamp string for embedding in the IDO filter.
    """
    shifted = value + timedelta(hours=SYTELINE_UTC_OFFSET_HOURS)
    return shifted.strftime("%Y-%m-%d %H:%M:%S")


def load_ido_records(job: Dict[str, Any], since: datetime) -> List[Dict[str, Any]]:
    """Stream all records for one entity from the IDO REST API since a watermark.

    Args:
        job: Ingest job dict from the control table.
        since: Lower bound for the incremental window (inclusive).

    Returns:
        Raw records as property-name -> value dicts.

    Raises:
        RuntimeError: On a non-200 IDO response.
    """
    entity = job["entity"]
    properties = sorted({
        src[2:-1]
        for field in job["fields"]
        for src in field["Sources"]
        if src.startswith("P(") and src.endswith(")")
    })
    filter_clause = f"{job['watermark_property']} >= '{format_syteline_timestamp(since)}'"
    if job["filter"]:
        filter_clause = f"({filter_clause}) AND ({job['filter']})"

    # The REST response carries Items / Bookmark / MoreRowsExist (PascalCase); the
    # Bookmark feeds the next page until MoreRowsExist goes false.
    url = f"{ION_API_BASE}/{ION_TENANT}/{ION_IDO_SUITE}/IDORequestService/ido/load/{entity}"
    records: List[Dict[str, Any]] = []
    bookmark: Optional[str] = None
    while True:
        params = {
            "properties": ",".join(properties),
            "filter": filter_clause,
            "recordCap": IDO_PAGE_SIZE,
            "loadType": "NEXT",
        }
        if job["order_by"]:
            params["orderBy"] = job["order_by"]
        if bookmark:
            params["bookmark"] = bookmark
        response = ion_get(url, params)
        if response.status_code != 200:
            raise RuntimeError(
                f"IDO load failed for {entity}: HTTP {response.status_code} - "
                f"{response.text[:300]}"
            )
        payload = response.json()
        records.extend(payload.get("Items", []))
        if not payload.get("MoreRowsExist"):
            break
        bookmark = payload.get("Bookmark")
        if not bookmark:
            break
        # Only multi-page pulls (backfills) narrate; steady-state increments stay quiet.
        print(f"    {entity}: {len(records):,} record(s) fetched so far...", flush=True)
    return records


# IDO date shapes normalized to ISO inside the transform - REST and SOAP surfaces return
# different date formats, and both must land as "s"-format ISO so downstream typed
# parsing (e.g. KQL todatetime()) never receives unparseable values.
_IDO_DATE_FORMATS = ("%Y%m%d %H:%M:%S.%f", "%m/%d/%Y %I:%M:%S %p")


def normalize_ido_value(value: Any) -> str:
    """Convert one raw IDO property value to its output string.

    Datetime-shaped strings normalize to ISO ("2026-06-03T13:02:03"); everything else
    passes through as str. None becomes "".

    Args:
        value: Raw property value from the IDO response.

    Returns:
        The normalized string value.
    """
    if value is None:
        return ""
    text = str(value)
    for fmt in _IDO_DATE_FORMATS:
        try:
            return datetime.strptime(text, fmt).strftime("%Y-%m-%dT%H:%M:%S")
        except ValueError:
            continue
    return text


def transform_record(record: Dict[str, Any], job: Dict[str, Any]) -> Dict[str, str]:
    """Map a raw IDO record to the job's output field shape.

    Each output field concatenates its sources: ``P(Name)`` resolves to the IDO property,
    anything else is a literal. All values are emitted as strings - pair with all-string
    landing tables and type downstream.

    Args:
        record: Raw IDO record (property name -> value).
        job: Ingest job dict.

    Returns:
        Output-field-name -> string-value dict - one row for the landing table.
    """
    output: Dict[str, str] = {}
    for field in job["fields"]:
        pieces = []
        for src in field["Sources"]:
            if src.startswith("P(") and src.endswith(")"):
                pieces.append(normalize_ido_value(record.get(src[2:-1])))
            else:
                pieces.append(src)
        output[field["OutputFieldName"]] = "".join(pieces)
    return output

## Transport: Kusto streaming ingestion

`POST {cluster}/v1/rest/ingest/{database}/{table}?streamFormat=MultiJSON` against the
Eventhouse's **engine (query) host** - *not* an `ingest-` host, which is queued ingestion -
carrying an Entra token for the Kusto audience. The body is newline-delimited JSON, one
object per record. That is the whole protocol: no envelope, no schema registry, no SDK.

**No ingestion mapping is required.** `transform_record` keys every payload by
`OutputFieldName`, which is the target column name, and every landing column is `string` -
so `MultiJSON` auto-maps by name. The REST documentation lists `mappingName` as mandatory
for JSON formats; it is not, for this shape (verified against a table created minutes
earlier).

**The response is the point.** Streaming ingestion does not answer until the rows are
*committed and queryable*, and the response carries `ConsumedRecordsCount` - located **by
column name, never by position**, because the receipt carries other columns whose order is
not contractual. A count short of what the batch held fails that run, at that batch, naming
that entity. There is no state in which this transport accepts bytes and drops them.

**Schema drift is handled by the engine, not gated.** A field the table lacks is ignored;
a column the payload omits lands empty. Both still count as consumed, so neither reads as a
failure here - the shape contract is the table's DDL. That is why `DRY_RUN` prints a sample
row: it is the only record of the sent shape.

### Two hazards this path has and the old one did not

**The streaming-ingestion schema cache.** Kusto caches each table's schema on the
Eventhouse nodes and **DDL does not invalidate it**. After a `.create-merge` adds a column,
ingests keep writing against the cached shape: the new column lands empty, with no error
anywhere. The remedy is `.clear table <T> cache streamingingestion schema`. `post_batch`
runs it itself on an `EntityNotFound` - once per send, then retries - so a table the
service has not caught up with recovers unattended. **The added-column case is still yours
to run by hand**: it produces no error to trigger on, which is exactly what makes it the
dangerous one.

**The cold-table `520`.** A table created moments earlier answers `520 Internal service
error` to every ingest for up to ~80 seconds. The retry budget covers it. The trap is that
`520` is also what a rename blocked by a mirroring policy returns, with the same misleading
"retry with backoff" advice - so a `520` that outlives the budget is worth reading the
nested `@message` for rather than retrying harder.

### Why this replaced a CloudEvents / Eventstream producer

This notebook used to wrap each row as a **CloudEvents 1.0** event and produce it to a
schema-associated Eventstream custom endpoint, which routed to an Eventhouse via
`tableName: {CloudEventType}_{CloudEventSchemaVersion}`. That is worth understanding before
reaching for the same shape, because the reasoning generalizes past this source:

- **Event Hubs acknowledges bytes, not delivery.** A `201` from the custom endpoint said
  the stream had the event and nothing about whether it reached the table. Three conditions
  ack and then silently drop: a schema set whose catalog has no entry for the
  `(type, version)` sent, a stale `dataschema` version, and an unwired or failed
  destination. No error surfaced at the producer, in the stream, or at the destination.
  Every defensive layer the old version carried - an endpoint probe, a schema-registry
  catalog pre-check, an eventstream topology check - existed to compensate for that one
  property. None of them defends against anything streaming ingestion can do.
- **A notebook producer uses none of what an Eventstream is for.** Fan-out to several
  destinations: there is one. In-flight transformation: none - the transform is above, in
  this notebook. Unbounded streams from many unknown producers: there is one, scheduled,
  pulling bounded windows. An Eventstream carrying one source to one destination with no
  operators is a hop, not a feature - and it *added* latency, since it still landed through
  queued ingestion under a batching policy.
- **The registry gate failed by dropping.** It did enforce a contract, by silently
  discarding what failed it. A gate whose failure output is indistinguishable from success
  is worse than no gate: it converts a loud misconfiguration into quiet data loss. Enforce
  the same contract in CI, at author time, in front of a human.
- **Publishing a per-schema Eventstream destination has returned `422` since September
  2026.** Already-wired entities keep running; no new one can be registered. Anyone
  adopting the schema-associated shape today should confirm that is fixed first.

**What was genuinely given up:** fan-out (unused here), and buffering across an outage
longer than the source's own retention. The second is covered by the control plane rather
than the transport - a failed send does not advance the watermark, so the next run re-pulls
the same window.

**When an Eventstream is still right:** many unknown producers, fan-out to several
destinations, in-flight transformation, or replay. If a second consumer of these same rows
appears, weigh a Kusto update policy or a second POST before reintroducing the hop.

Removing it took this notebook from 13 parameters to 8 and dropped one Key Vault secret, the
`azure-eventhub` self-install, and the AMQP producer.

In [ ]:
def target_table(job: Dict[str, Any]) -> str:
    """Compose the landing table name for one entity.

    ``<SourceObjectName>_<SchemaVersion>``. The convention is inherited - it was once a
    routing expression on an Eventstream destination (``{CloudEventType}_{CloudEventSchemaVersion}``)
    rather than a choice this notebook made. It is kept because the tables are live under
    those names, and because versioning the table rather than a registry entry is what lets
    a breaking shape change land beside the old one instead of on top of it.

    Args:
        job: Ingest job dict from the control table.

    Returns:
        Unquoted table name.
    """
    return f"{job['entity']}_{job['schema_version']}"


def print_sample_row(job: Dict[str, Any], payload: Dict[str, str]) -> None:
    """Print one representative record, as the ingest request carries it.

    Debugging aid for the quietest failure mode on this path: streaming ingestion ignores a
    field the table does not have and leaves a column the payload omits empty, and both
    still count as consumed - so neither reads as a failure anywhere else. One sample per
    run makes the sent shape recoverable after the fact.

    Args:
        job: Ingest job dict (supplies the target table name).
        payload: The transformed record to print.
    """
    body = json.dumps(payload, ensure_ascii=False)
    # No envelope and no attributes to show: the body IS the row. What is worth showing
    # instead is which table it lands in, because that name is the only routing decision
    # left on this path.
    print(f"  -- sample row - {job['entity']} (first record) --")
    print(f"    MultiJSON line, one per record, into ['{target_table(job)}'] - properties "
          "auto-map to same-named columns, no ingestion mapping")
    print(f"    body ({len(payload):,} field(s), {len(body.encode('utf-8')):,} bytes):")
    truncated = "" if len(body) <= SAMPLE_ROW_MAX_CHARS else f" ... [of {len(body):,} chars]"
    print(f"      {body[:SAMPLE_ROW_MAX_CHARS]}{truncated}")


def kusto_mgmt(command: str, database: str) -> List[Dict[str, Any]]:
    """Run one KQL management command and return its first result table as row dicts.

    The management endpoint (``/v1/rest/mgmt``), not the query one - a control command
    posted to ``/v1/rest/query`` is refused. Same host and same Entra audience as the
    streaming ingest, so nothing new has to resolve or be granted for this to work.

    Args:
        command: Control command text, e.g. ``.clear table ['T'] cache ...``.
        database: KQL database to run it against.

    Returns:
        Rows of the first result table, each keyed by column name. Empty when the response
        carried no tables.

    Raises:
        RuntimeError: On a non-200 response.
    """
    token = notebookutils.credentials.getToken(KUSTO_TOKEN_AUDIENCE)
    response = requests.post(
        f"{KUSTO_URI.rstrip('/')}/v1/rest/mgmt",
        json={"db": database, "csl": command},
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        timeout=60,
    )
    if response.status_code != 200:
        raise RuntimeError(f"Kusto HTTP {response.status_code}: {response.text[:300]}")
    tables = response.json().get("Tables") or []
    if not tables:
        return []
    names = [col.get("ColumnName") for col in tables[0].get("Columns") or []]
    return [dict(zip(names, row)) for row in tables[0].get("Rows") or []]


def clear_streaming_schema_cache(table: str, database: str) -> str:
    """Clear one table's streaming-ingestion schema cache. Empty string means it worked.

    Scoped to the one table, not the database: the caller knows exactly which table failed,
    and the database-wide form would drop the cache for every healthy table to fix one.
    That form stays what it is - the thing to run by hand after bulk DDL.

    **The result is the rows, not the status code.** The command reports per node, and a
    node can come back ``Failed`` inside an HTTP 200 - so a caller that checked only the
    status would claim a heal it never got, which is worse than not trying at all.

    Args:
        table: Target table name, unquoted.
        database: KQL database holding it.

    Returns:
        Empty string when every node reported ``Succeeded``; otherwise a short reason for
        the caller to put in front of its own error. Never raises - a failure here must not
        replace the ingest error that prompted it, which says considerably more.
    """
    command = f".clear table ['{table}'] cache streamingingestion schema"
    try:
        rows = kusto_mgmt(command, database)
    except Exception as exc:
        return f"could not run `{command}`: {exc}"
    if not rows:
        return f"`{command}` returned no node rows"
    failed = [str(row.get("NodeId")) for row in rows if row.get("Status") != "Succeeded"]
    if failed:
        return (f"`{command}` failed on {len(failed)} of {len(rows)} node(s): "
                f"{', '.join(failed[:5])}")
    return ""


def kusto_stream_ingest(
    database: str,
    table: str,
    payloads: List[Dict[str, str]],
    context: str,
) -> int:
    """Write payloads straight into one Kusto table by streaming ingestion.

    ``POST /v1/rest/ingest/{db}/{table}`` against the engine host ``KUSTO_URI`` names, with
    an Entra token for the Kusto audience. Each payload's keys are the target table's
    column names, so ``streamFormat=MultiJSON`` auto-maps by name and no ingestion mapping
    is required.

    The call returns only once the rows are committed and queryable, and reports
    ``ConsumedRecordsCount`` - so a call that returns is the delivery proof, and a short
    count fails at the batch that lost rows.

    Args:
        database: KQL database holding the target table.
        table: Target table name, already composed by the caller.
        payloads: Records to send.
        context: Name used in error text to say what was being sent. Only surfaces in
            exceptions.

    Returns:
        Number of records ingested.

    Raises:
        RuntimeError: On a non-200 response, a ConsumedRecordsCount short of what the batch
            held, or a single record too large for a batch on its own.
    """
    post_url = (
        f"{KUSTO_URI.rstrip('/')}/v1/rest/ingest/"
        f"{urllib.parse.quote(database)}/{urllib.parse.quote(table)}"
        "?streamFormat=MultiJSON"
    )
    headers = {
        "Authorization": f"Bearer {notebookutils.credentials.getToken(KUSTO_TOKEN_AUDIENCE)}",
        "Content-Type": "application/json",
    }
    token_minted_at = time.monotonic()
    # One self-heal per call, not per batch: a table that is genuinely absent would
    # otherwise clear and retry once for every batch in the window.
    schema_cache_cleared = False

    def consumed_count(body: Dict[str, Any]) -> Optional[int]:
        """Read ConsumedRecordsCount out of the ingest response, or None if absent.

        Located by column name rather than position: the receipt carries other columns and
        their order is not contracted anywhere. None means the shape changed, which is not
        itself a delivery failure - the caller treats it as "no receipt", not a short count.
        """
        for result_table in body.get("Tables") or []:
            names = [col.get("ColumnName") for col in result_table.get("Columns") or []]
            if "ConsumedRecordsCount" not in names:
                continue
            rows = result_table.get("Rows") or []
            if rows:
                return int(rows[0][names.index("ConsumedRecordsCount")])
        return None

    def post_batch(session: requests.Session, rows: List[str]) -> None:
        """POST one batch, retrying transient failures and self-healing a stale cache."""
        # Token checked per batch, not per record: a batch is the only thing that spends
        # it, and this loop can run for a long time behind a large window.
        nonlocal token_minted_at, schema_cache_cleared
        if time.monotonic() - token_minted_at > KUSTO_TOKEN_REFRESH_AFTER_SECONDS:
            headers["Authorization"] = (
                f"Bearer {notebookutils.credentials.getToken(KUSTO_TOKEN_AUDIENCE)}"
            )
            token_minted_at = time.monotonic()
        body = "\n".join(rows).encode("utf-8")
        response = None
        heal_note = ""
        for attempt in range(KUSTO_INGEST_ATTEMPTS):
            response = session.post(post_url, data=body, headers=headers, timeout=120)
            if response.status_code == 200:
                break
            retryable = response.status_code in KUSTO_RETRY_STATUS
            # EntityNotFound is a 400 and so not retryable above, but it has one remedy
            # worth spending an attempt on: the table may exist and simply be missing from
            # the streaming-ingestion schema cache, which DDL does not invalidate and the
            # service refreshes on its own only after several minutes. A clear that does
            # not report success on every node earns nothing - the original failure is the
            # more useful error, and a half-cleared cache is not a reason to send again.
            if (not retryable and not schema_cache_cleared
                    and "EntityNotFound" in response.text):
                schema_cache_cleared = True
                reason = clear_streaming_schema_cache(table, database)
                if reason:
                    heal_note = f"\n    Could not self-heal: {reason}."
                else:
                    heal_note = (f"\n    Cleared ['{table}'] streaming-ingestion schema "
                                 "cache and retried; still not ingestable.")
                    print(f"    ['{table}'] answered EntityNotFound - cleared its "
                          "streaming-ingestion schema cache, retrying")
                    retryable = True
            if not retryable or attempt == KUSTO_INGEST_ATTEMPTS - 1:
                raise RuntimeError(
                    f"Kusto streaming ingest failed for {context} into ['{table}']: "
                    f"HTTP {response.status_code} - {response.text[:300]}"
                    + heal_note
                    + (KUSTO_TABLE_MISSING_HINT.format(table=table)
                       if "EntityNotFound" in response.text else "")
                )
            backoff = min(KUSTO_RETRY_BASE_DELAY * (2 ** attempt), KUSTO_RETRY_MAX_DELAY)
            time.sleep(backoff + random.uniform(0, 1))
        consumed = consumed_count(response.json() or {})
        if consumed is not None and consumed != len(rows):
            raise RuntimeError(
                f"Kusto streaming ingest short-counted for {context} into ['{table}']: "
                f"sent {len(rows):,} record(s), consumed {consumed:,}"
            )

    sent = 0
    batch: List[str] = []
    batch_bytes = 0
    # One Session per call: keep-alive reuses a single TCP+TLS connection across every
    # batch POST instead of paying a fresh handshake per request.
    with requests.Session() as http_session:
        for payload in payloads:
            line = json.dumps(payload, ensure_ascii=False)
            line_bytes = len(line.encode("utf-8")) + 1  # +1 joining newline
            if line_bytes > KUSTO_STREAM_LIMIT_BYTES:
                raise RuntimeError(
                    f"Single record for {context} exceeds the streaming ingest size limit "
                    f"({KUSTO_STREAM_LIMIT_BYTES:,} bytes)"
                )
            if batch and batch_bytes + line_bytes > KUSTO_STREAM_LIMIT_BYTES:
                post_batch(http_session, batch)
                sent += len(batch)
                batch, batch_bytes = [], 0
            batch.append(line)
            batch_bytes += line_bytes
        if batch:
            post_batch(http_session, batch)
            sent += len(batch)
    return sent


def send_records(job: Dict[str, Any], payloads: List[Dict[str, str]]) -> int:
    """Write all payloads for one entity into its landing table.

    Every landing column is string and ``transform_record`` has already keyed each payload
    by OutputFieldName - which is the column name - so the records go out as they are.

    Args:
        job: Ingest job dict.
        payloads: Transformed records to send.

    Returns:
        Number of records committed, as reported by the ingest receipt.
    """
    return kusto_stream_ingest(
        database=KUSTO_DATABASE,
        table=target_table(job),
        payloads=payloads,
        context=job["entity"],
    )

## Run (soft fail per entity)

One entity failing must not block the rest; failures keep their old watermark and retry
the same window next run. Watermarks are captured at run start and persisted only on
success, so overlap duplicates are possible by design - deduplicate downstream (e.g. KQL
`arg_max` materialized views or Delta merge). `DRY_RUN` reports what *would* be sent.

**Advancing the watermark as soon as the send returns is sound on this transport, and was
not on the old one.** Under the Eventstream, "success" meant Event Hubs had accepted the
bytes - a window could be marked done and then silently dropped at the registry gate, which
is what drove designs like a two-phase watermark that parks a pending value and promotes it
only after counting rows in the destination. Streaming ingestion has no
accepted-but-not-landed state: the call returns once the rows are committed, and a short
`ConsumedRecordsCount` raises. A send that returns is the proof, so one phase is enough.

In [ ]:
known_entities = {j["entity"] for j in job_definitions}
unknown_filters = [e for e in ENTITY_FILTER if e not in known_entities]
if unknown_filters:
    raise RuntimeError(f"ENTITY_FILTER names match no ingest control record: {unknown_filters}")

sample_printed = False

for job in job_definitions:
    entity = job["entity"]
    if ENTITY_FILTER and entity not in ENTITY_FILTER:
        results["skipped"].append({"name": entity, "reason": "not in ENTITY_FILTER"})
        continue
    try:
        started = time.monotonic()
        since = job["last_watermark"] or datetime.fromisoformat(DEFAULT_WATERMARK)
        run_started = datetime.now(timezone.utc).replace(tzinfo=None)
        raw_records = load_ido_records(job, since)
        payloads = [transform_record(r, job) for r in raw_records]

        if DRY_RUN:
            print(f"[DRY_RUN] {entity}: {len(payloads):,} record(s) since "
                  f"{since:%Y-%m-%d %H:%M} -> ['{target_table(job)}'] "
                  f"({time.monotonic() - started:.0f}s)")
            # One sample for the whole run, not per entity. Streaming ingestion silently
            # tolerates both directions of schema drift, so this print is the only record
            # of the shape actually being sent.
            if payloads and not sample_printed:
                print_sample_row(job, payloads[0])
                sample_printed = True
        else:
            sent = send_records(job, payloads)
            set_watermark(entity, run_started)
            elapsed = int(time.monotonic() - started)
            log_run(
                entity,
                "SUCCESS" if sent > 0 else "NOROWS",
                records_fetched=len(raw_records),
                records_sent=sent,
                duration_seconds=elapsed,
                watermark=run_started,
            )
            print(f"{entity}: committed {sent:,} record(s) to ['{target_table(job)}'] "
                  f"in {elapsed}s; watermark -> {run_started:%Y-%m-%d %H:%M:%S}")
        results["succeeded"].append(entity)
    except Exception as exc:
        results["failed"].append({"name": entity, "error": str(exc)})
        if not DRY_RUN:
            log_run(entity, "FAILED",
                    duration_seconds=int(time.monotonic() - started),
                    failure_message=str(exc))

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────────
print("── Summary ───────────────────────────────────")
print(f"  Mode:      {'DRY_RUN' if DRY_RUN else 'LIVE'}")
print(f"  Succeeded: {len(results['succeeded'])}")
print(f"  Skipped:   {len(results['skipped'])}")
print(f"  Failed:    {len(results['failed'])}")
for item in results["failed"]:
    print(f"    - {item['name']}: {item['error']}")

if STRICT and results["failed"]:
    raise RuntimeError(f"{len(results['failed'])} entity load(s) failed (STRICT mode)")

# Exit value for a pipeline: @activity('<NotebookActivity>').output.result.exitValue
exit_value = json.dumps({
    "mode": "DRY_RUN" if DRY_RUN else "LIVE",
    "runId": RUN_ID,
    "succeeded": len(results["succeeded"]),
    "skipped": len(results["skipped"]),
    "failed": len(results["failed"]),
    "failedEntities": [f["name"] for f in results["failed"]],
})
# exit() halts execution by raising an internal NotebookExit exception, so an interactive
# run renders this cell as errored even when everything succeeded. Only pipeline runs need
# the exit (it is what surfaces exitValue); interactive runs print the same JSON instead.
# Must stay outside any try/except or the exit is swallowed.
try:
    _is_pipeline_run = bool(notebookutils.runtime.context.get("isForPipeline"))
except Exception:
    _is_pipeline_run = bool(PIPELINE_RUN_ID)
if _is_pipeline_run and hasattr(notebookutils.notebook, "exit"):
    notebookutils.notebook.exit(exit_value)
else:
    print(exit_value)